# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 60 packages in 1.05s
Checked 60 packages in 3ms


## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [2]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), and AzureOpenAI
from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr


c:\Users\Test\OneDrive\Desktop\Everything\Classes\Fall 2026 Classes\Agentic AI\Lab 5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 4d7765fb


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [72]:
persona = (
    "Lumi, A relaxed, chill and helpful study tutor. Is not too hard on you for mistakes and helps ypu build your understanding."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")

Lumi, A relaxed, chill and helpful study tutor. Is not too hard on you for mistakes and helps ypu build your understanding.
Notes on Basketball 
• A “3 point shot” is a basket that is scored from beyond the 3-point line 
• A “free throw” is the compensation free shots from the free throw line after a 
shooting-foul is committed 
• A “Dunk” is when a player slams the basketball directly through the basket. 
• A “Center” is the 5th position in basketball, generally tasked with rim protecting, 
rebounds, and inside-the-post scoring.  
• A “Point Guard” is the playmaker of the team, responsible for distributing the ball  ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [73]:
system_prompt = f"""
# Your role. 

You are {persona}

You are to test and quiz a student on the study notes that have been provided.

# Context

Here are the notes you are going to quiz the student on.

{notes}

# Rules

Go through the notes one at a time and ask a question to the student based on a bullet point in the notes, wait for the students answer before
asking another.

After the student gives his answer, tell him if he was right or wrong, and give a short explanation on either why he is right or why he was wrong.

Anytime the student is wrong, record it as a missed question, and if they say "I dont know", then skip the question and move on to the next. 

ONLY ask questions about the notes.
"""


display(Markdown(system_prompt))


# Your role. 

You are Lumi, A relaxed, chill and helpful study tutor. Is not too hard on you for mistakes and helps ypu build your understanding.

You are to test and quiz a student on the study notes that have been provided.

# Context

Here are the notes you are going to quiz the student on.

Notes on Basketball 
• A “3 point shot” is a basket that is scored from beyond the 3-point line 
• A “free throw” is the compensation free shots from the free throw line after a 
shooting-foul is committed 
• A “Dunk” is when a player slams the basketball directly through the basket. 
• A “Center” is the 5th position in basketball, generally tasked with rim protecting, 
rebounds, and inside-the-post scoring.  
• A “Point Guard” is the playmaker of the team, responsible for distributing the ball 
and orchestrating the offense. 

# Rules

Go through the notes one at a time and ask a question to the student based on a bullet point in the notes, wait for the students answer before
asking another.

After the student gives his answer, tell him if he was right or wrong, and give a short explanation on either why he is right or why he was wrong.

Anytime the student is wrong, record it as a missed question, and if they say "I dont know", then skip the question and move on to the next. 

ONLY ask questions about the notes.


## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [74]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Lets begin the quiz!"},
]

# TODO: call azure_client.chat.completions.create(...) with model=DEPLOYMENT and messages=messages,
# then display the result as Markdown
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
display(Markdown(response.choices[0].message.content))


Awesome — let’s start easy.

Question 1: What is a “3 point shot” in basketball?

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [75]:
def chat(message, history):
    # TODO: build the messages list (system prompt + history + new user message)
    # TODO: call the model and return response.choices[0].message.content
    pass
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    return response.choices[0].message.content

In [76]:
chat("I'm ready, let's start the quiz", [])

'Awesome — let’s go one at a time.\n\n**Question 1:**  \nWhat is a **3-point shot** in basketball?'

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [79]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [80]:
def track_score_tool(correct):
    print(f"Tool called to log when a question is answered correctly: {correct}")
    with open("correct_answers.txt", "a", encoding="utf-8") as f:
        f.write(correct + "\n")
    return "Correct Answer logged"

   

In [81]:
track_score_tool("Correct")  # quick manual test of your tool

Tool called to log when a question is answered correctly: Correct


'Correct Answer logged'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [82]:
track_score_tool_json = {
    "name": "track_score_tool",
    "description": "Use this tool to log when a student has answered a question correctly",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question or concept the student answered correctly"}
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": track_score_tool_json}]

## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.

In [83]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        question = json.loads(tool_call.function.arguments).get("value")
        track_score_tool(str(question))
        messages.append(message_obj)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": "Correct Answer logged"})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

In [84]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [85]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            question = json.loads(tool_call.function.arguments).get("question")
            track_score_tool(question)
            messages.append({"role": "tool", "tool_call_id": tool_call.id,
                "content": "Missed question logged"})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)
    # TODO: while response.choices[0].finish_reason == "tool_calls":
    #     - message_obj = response.choices[0].message
    #     - messages.append(message_obj)
    #     - for tool_call in message_obj.tool_calls:
    #           - parse arguments, call your tool, append the "tool" role message
    #     - response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 12. Try your finished Persona Agent
Same one line as before.

In [86]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make
   sense for this topic?

      I chose basketball trivia, and I gave my Persona the ability to log when the student got an answer right. I chose this because I like basketball and know enough about it to write my own notes, and the tool seemed fitting to track what questions the student got right.

2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What,
   if anything, did you have to think about differently?

      I honestly know there was some difference, but not different enough for me to directly put a finger on it. I would say the Persona's responsibility, since this topic is more trivia than a true test. 

3. What would break if you asked your Persona a question completely unrelated to your notes? Did
   your system prompt handle that the way you wanted?

      Nothing "broke" persay, but it responds with "lets stick to whats on the notes", then continues with the question. I would say its ability to be a flexible chat bot outside of the notes breaks.
      
4. If you added a second tool to this agent, what would it be, and what would change in Steps
   9–11 to support it?

   for step 9 I would have to define a second tool in the tools list, for step 10 I would need some sort of statement to separate the logic of tool 1 and tool 2. Then, for step 11, I dont think much would have to change. 

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.